# Lesson 05: Going Modular with PyTorch

This notebook covers:
- Organizing PyTorch code into reusable modules
- Creating Python scripts for:
  - Data loading (`data_setup.py`)
  - Model architecture (`model_builder.py`)
  - Training/testing loops (`engine.py`)
  - Utility functions (`utils.py`)
  - Main training script (`train.py`)
- Writing production-ready, maintainable code
- Best practices for project structure

**Why go modular?**
- **Reusability**: Write once, use many times
- **Maintainability**: Easy to find and fix bugs
- **Collaboration**: Others can understand your code
- **Scalability**: Easy to add new features
- **Testing**: Easier to unit test individual components

## Module 1: data_setup.py - Data Loading Functions

This module handles all data loading logic:
- Creating datasets from directories
- Setting up DataLoaders with optimal settings
- Returning class names for reference

In [ ]:
%%writefile going_modular/data_setup.py
"""
Contains functionality for creating PyTorch DataLoaders for 
image classification data.

This module provides a single function that handles the entire
data loading pipeline, making it easy to load datasets consistently
across different projects.
"""
import os

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Set number of workers to CPU count for optimal performance
# More workers = faster data loading (up to a point)
NUM_WORKERS = os.cpu_count()

def create_dataloaders(
    train_dir: str, 
    test_dir: str, 
    transform: transforms.Compose, 
    batch_size: int, 
    num_workers: int=NUM_WORKERS
):
    """
    Creates training and testing DataLoaders.

    Takes in a training directory and testing directory path and turns
    them into PyTorch Datasets and then into PyTorch DataLoaders.

    Args:
        train_dir: Path to training directory.
        test_dir: Path to testing directory.
        transform: torchvision transforms to perform on training and testing data.
        batch_size: Number of samples per batch in each of the DataLoaders.
        num_workers: An integer for number of workers per DataLoader.

    Returns:
        A tuple of (train_dataloader, test_dataloader, class_names).
        Where class_names is a list of the target classes.
        
    Example usage:
        train_dataloader, test_dataloader, class_names = \
            create_dataloaders(train_dir=path/to/train_dir,
                             test_dir=path/to/test_dir,
                             transform=some_transform,
                             batch_size=32,
                             num_workers=4)
    """
    # Use ImageFolder to create dataset(s)
    # ImageFolder expects: root/class_name/image.jpg structure
    train_data = datasets.ImageFolder(train_dir, transform=transform)
    test_data = datasets.ImageFolder(test_dir, transform=transform)

    # Get class names from the dataset
    # ImageFolder automatically extracts these from directory names
    class_names = train_data.classes

    # Turn images into DataLoaders
    train_dataloader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,  # Shuffle training data for better learning
        num_workers=num_workers,  # Parallel data loading
        pin_memory=True,  # Speed up data transfer to GPU
    )
    
    test_dataloader = DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=False,  # Don't need to shuffle test data
        num_workers=num_workers,
        pin_memory=True,
    )

    return train_dataloader, test_dataloader, class_names

## Module 2: model_builder.py - Model Architecture

This module defines model architectures:
- TinyVGG CNN for image classification
- Can easily add more model architectures
- Keeps all model code in one place

In [ ]:
%%writefile going_modular/model_builder.py
"""
Contains PyTorch model code to instantiate a TinyVGG model.

This module provides model architectures that can be imported
and used in different training scripts. Add new models here
as you experiment with different architectures.
"""
import torch
from torch import nn 

class TinyVGG(nn.Module):
    """
    Creates the TinyVGG architecture.

    Replicates the TinyVGG architecture from the CNN explainer website in PyTorch.
    See the original architecture here: https://poloclub.github.io/cnn-explainer/
    
    Architecture:
    - Conv Block 1: Conv2d -> ReLU -> Conv2d -> ReLU -> MaxPool2d
    - Conv Block 2: Conv2d -> ReLU -> Conv2d -> ReLU -> MaxPool2d
    - Classifier: Flatten -> Linear
    
    Args:
        input_shape: An integer indicating number of input channels.
                    (3 for RGB, 1 for grayscale)
        hidden_units: An integer indicating number of hidden units between layers.
        output_shape: An integer indicating number of output units.
                     (number of classes for classification)
    """
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int) -> None:
        super().__init__()
        
        # First convolutional block
        # Detects low-level features (edges, colors, textures)
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(
                in_channels=input_shape, 
                out_channels=hidden_units, 
                kernel_size=3,  # 3x3 filter
                stride=1,  # Move 1 pixel at a time
                padding=0  # No padding (image gets smaller)
            ),  
            nn.ReLU(),  # Non-linear activation
            nn.Conv2d(
                in_channels=hidden_units, 
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=0
            ),
            nn.ReLU(),
            # MaxPool reduces spatial dimensions by half
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        # Second convolutional block
        # Detects higher-level features (patterns, shapes)
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        # Classifier head
        # Converts learned features to class predictions
        self.classifier = nn.Sequential(
            nn.Flatten(),  # Flatten 3D feature maps to 1D
            
            # Where did this in_features shape come from?
            # Each layer compresses the input:
            # Input: 64x64 -> Conv(no padding): 62x62 -> Conv: 60x60 -> MaxPool: 30x30
            # -> Conv: 28x28 -> Conv: 26x26 -> MaxPool: 13x13
            # With hidden_units filters: hidden_units * 13 * 13
            nn.Linear(
                in_features=hidden_units*13*13,
                out_features=output_shape
            )
        )
    
    def forward(self, x: torch.Tensor):
        """
        Forward pass through the network.
        
        Args:
            x: Input tensor of shape (batch_size, channels, height, width)
            
        Returns:
            Output tensor of shape (batch_size, output_shape)
        """
        # Pass through each block sequentially
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        x = self.classifier(x)
        return x
        
        # Alternative: leverage operator fusion for potential speedup
        # return self.classifier(self.conv_block_2(self.conv_block_1(x)))

## Module 3: engine.py - Training and Testing Logic

This module contains the core training loop:
- `train_step()`: Single training epoch
- `test_step()`: Single evaluation epoch  
- `train()`: Complete training loop with both steps

This is the "engine" that powers model training.

In [ ]:
%%writefile going_modular/engine.py
"""
Contains functions for training and testing a PyTorch model.

This module provides the core training loop functionality that can be
reused across different projects and model architectures.
"""
import torch

from tqdm.auto import tqdm  # Progress bar
from typing import Dict, List, Tuple

def train_step(
    model: torch.nn.Module, 
    dataloader: torch.utils.data.DataLoader, 
    loss_fn: torch.nn.Module, 
    optimizer: torch.optim.Optimizer,
    device: torch.device
) -> Tuple[float, float]:
    """
    Trains a PyTorch model for a single epoch.

    Turns a target PyTorch model to training mode and then
    runs through all of the required training steps (forward
    pass, loss calculation, optimizer step).

    Args:
        model: A PyTorch model to be trained.
        dataloader: A DataLoader instance for the model to be trained on.
        loss_fn: A PyTorch loss function to minimize.
        optimizer: A PyTorch optimizer to help minimize the loss function.
        device: A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
        A tuple of training loss and training accuracy metrics.
        In the form (train_loss, train_accuracy). For example:
        (0.1112, 0.8743)
    """
    # Put model in train mode
    # This enables dropout, batch norm, etc. for training
    model.train()
    
    # Setup train loss and train accuracy values
    train_loss, train_acc = 0, 0
    
    # Loop through data loader data batches
    for batch, (X, y) in enumerate(dataloader):
        # Send data to target device
        X, y = X.to(device), y.to(device)

        # 1. Forward pass
        y_pred = model(X)

        # 2. Calculate and accumulate loss
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()  # .item() converts to Python number

        # 3. Optimizer zero grad
        # Clear gradients from previous batch
        optimizer.zero_grad()

        # 4. Loss backward
        # Compute gradients via backpropagation
        loss.backward()

        # 5. Optimizer step
        # Update model parameters using gradients
        optimizer.step()

        # Calculate and accumulate accuracy metric across all batches
        # Get predicted class (highest probability)
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)

    # Adjust metrics to get average loss and accuracy per batch
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc

def test_step(
    model: torch.nn.Module, 
    dataloader: torch.utils.data.DataLoader, 
    loss_fn: torch.nn.Module,
    device: torch.device
) -> Tuple[float, float]:
    """
    Tests a PyTorch model for a single epoch.

    Turns a target PyTorch model to "eval" mode and then performs
    a forward pass on a testing dataset.

    Args:
        model: A PyTorch model to be tested.
        dataloader: A DataLoader instance for the model to be tested on.
        loss_fn: A PyTorch loss function to calculate loss on the test data.
        device: A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
        A tuple of testing loss and testing accuracy metrics.
        In the form (test_loss, test_accuracy). For example:
        (0.0223, 0.8985)
    """
    # Put model in eval mode
    # This disables dropout, uses running stats for batch norm, etc.
    model.eval() 
    
    # Setup test loss and test accuracy values
    test_loss, test_acc = 0, 0
    
    # Turn on inference context manager
    # Disables gradient tracking for faster computation and less memory
    with torch.inference_mode():
        # Loop through DataLoader batches
        for batch, (X, y) in enumerate(dataloader):
            # Send data to target device
            X, y = X.to(device), y.to(device)
    
            # 1. Forward pass
            test_pred_logits = model(X)

            # 2. Calculate and accumulate loss
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()
            
            # Calculate and accumulate accuracy
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += ((test_pred_labels == y).sum().item()/len(test_pred_labels))
            
    # Adjust metrics to get average loss and accuracy per batch
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc

def train(
    model: torch.nn.Module, 
    train_dataloader: torch.utils.data.DataLoader, 
    test_dataloader: torch.utils.data.DataLoader, 
    optimizer: torch.optim.Optimizer,
    loss_fn: torch.nn.Module,
    epochs: int,
    device: torch.device
) -> Dict[str, List]:
    """
    Trains and tests a PyTorch model.

    Passes a target PyTorch model through train_step() and test_step()
    functions for a number of epochs, training and testing the model
    in the same epoch loop.

    Calculates, prints and stores evaluation metrics throughout.

    Args:
        model: A PyTorch model to be trained and tested.
        train_dataloader: A DataLoader instance for the model to be trained on.
        test_dataloader: A DataLoader instance for the model to be tested on.
        optimizer: A PyTorch optimizer to help minimize the loss function.
        loss_fn: A PyTorch loss function to calculate loss on both datasets.
        epochs: An integer indicating how many epochs to train for.
        device: A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
        A dictionary of training and testing loss as well as training and
        testing accuracy metrics. Each metric has a value in a list for 
        each epoch.
        In the form: {train_loss: [...],
                      train_acc: [...],
                      test_loss: [...],
                      test_acc: [...]} 
        For example if training for epochs=2: 
                     {train_loss: [2.0616, 1.0537],
                      train_acc: [0.3945, 0.3945],
                      test_loss: [1.2641, 1.5706],
                      test_acc: [0.3400, 0.2973]} 
    """
    # Create empty results dictionary
    results = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }
    
    # Loop through training and testing steps for a number of epochs
    for epoch in tqdm(range(epochs)):
        # Perform training step
        train_loss, train_acc = train_step(
            model=model,
            dataloader=train_dataloader,
            loss_fn=loss_fn,
            optimizer=optimizer,
            device=device
        )
        
        # Perform testing step
        test_loss, test_acc = test_step(
            model=model,
            dataloader=test_dataloader,
            loss_fn=loss_fn,
            device=device
        )
        
        # Print out what's happening
        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc:.4f}"
        )

        # Update results dictionary
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

    # Return the filled results at the end of the epochs
    return results

## Module 4: utils.py - Utility Functions

This module contains helper functions:
- Saving models
- Loading models  
- Plotting results
- Any other utility code

In [ ]:
%%writefile going_modular/utils.py
"""
Contains various utility functions for PyTorch model training and saving.

This module provides helper functions that don't fit neatly into other
modules but are still essential for the training workflow.
"""
import torch
from pathlib import Path

def save_model(
    model: torch.nn.Module,
    target_dir: str,
    model_name: str
):
    """
    Saves a PyTorch model to a target directory.

    Args:
        model: A target PyTorch model to save.
        target_dir: A directory for saving the model to.
        model_name: A filename for the saved model. Should include
                   either ".pth" or ".pt" as the file extension.
    
    Example usage:
        save_model(model=model_0,
                   target_dir="models",
                   model_name="05_going_modular_tinyvgg_model.pth")
    """
    # Create target directory if it doesn't exist
    target_dir_path = Path(target_dir)
    target_dir_path.mkdir(
        parents=True,  # Create parent directories if needed
        exist_ok=True  # Don't error if directory already exists
    )
    
    # Create model save path
    # Validate file extension
    assert model_name.endswith(".pth") or model_name.endswith(".pt"), \
        "model_name should end with '.pt' or '.pth'"
    model_save_path = target_dir_path / model_name

    # Save the model state_dict()
    # Only saves the parameters, not the model architecture
    print(f"[INFO] Saving model to: {model_save_path}")
    torch.save(
        obj=model.state_dict(),
        f=model_save_path
    )

## Module 5: train.py - Main Training Script

This is the script that ties everything together:
- Imports all other modules
- Sets hyperparameters
- Creates data, model, loss, optimizer
- Runs training
- Saves model

This is what you run from the command line!

In [ ]:
%%writefile going_modular/train.py
"""
Trains a PyTorch image classification model using device-agnostic code.

This is the main training script that brings together all the modular
components. It can be run from the command line:

    python train.py

Or imported and used in other scripts/notebooks.
"""

import os
import torch

# Import our custom modules
# These should all be in the same directory as this script
import data_setup, engine, model_builder, utils

from torchvision import transforms

# ============================================================================
# Setup hyperparameters
# ============================================================================
# These control the training process and model architecture
NUM_EPOCHS = 5  # How many complete passes through the data
BATCH_SIZE = 32  # Number of samples per batch
HIDDEN_UNITS = 10  # Number of neurons in hidden layers
LEARNING_RATE = 0.001  # Step size for parameter updates

# ============================================================================
# Setup directories
# ============================================================================
# These should point to your actual data directories
train_dir = "data/pizza_steak_sushi/train"
test_dir = "data/pizza_steak_sushi/test"

# ============================================================================
# Setup target device
# ============================================================================
# Use GPU if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================================
# Create transforms
# ============================================================================
# Define how to preprocess images
data_transform = transforms.Compose([
    transforms.Resize((64, 64)),  # Resize to consistent size
    transforms.ToTensor()  # Convert to PyTorch tensor
])

# ============================================================================
# Create DataLoaders with help from data_setup.py
# ============================================================================
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=data_transform,
    batch_size=BATCH_SIZE
)

# ============================================================================
# Create model with help from model_builder.py
# ============================================================================
model = model_builder.TinyVGG(
    input_shape=3,  # RGB images (3 color channels)
    hidden_units=HIDDEN_UNITS,
    output_shape=len(class_names)  # One output per class
).to(device)  # Move model to target device

# ============================================================================
# Set loss function and optimizer
# ============================================================================
# CrossEntropyLoss for multi-class classification
loss_fn = torch.nn.CrossEntropyLoss()

# Adam optimizer with specified learning rate
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

# ============================================================================
# Start training with help from engine.py
# ============================================================================
engine.train(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=NUM_EPOCHS,
    device=device
)

# ============================================================================
# Save the trained model with help from utils.py
# ============================================================================
utils.save_model(
    model=model,
    target_dir="models",
    model_name="05_going_modular_script_mode_tinyvgg_model.pth"
)

## Running the Training Script

Now we can run our modular training script from the command line!

In [ ]:
# Run the training script
# This executes train.py which imports and uses all our modules
!python going_modular/train.py

## Summary: Benefits of Going Modular

### 1. **Reusability**
- Use the same data loading code across multiple projects
- Share model architectures between experiments
- Reuse training loops with different models

### 2. **Maintainability**
- Easy to find and fix bugs (know exactly which file to check)
- Clear separation of concerns
- Easier to update individual components

### 3. **Collaboration**
- Others can understand your code structure
- Easy to divide work among team members
- Standard structure across projects

### 4. **Testing**
- Can unit test individual functions
- Easier to debug specific components
- Validate each piece independently

### 5. **Experimentation**
- Swap out components easily (try different models, optimizers, etc.)
- Keep experiments organized
- Compare results systematically

### Typical Project Structure:
```
project/
│
├── going_modular/
│   ├── __init__.py
│   ├── data_setup.py      # Data loading
│   ├── model_builder.py   # Model architectures
│   ├── engine.py          # Training/testing loops
│   ├── utils.py           # Helper functions
│   └── train.py           # Main training script
│
├── data/                  # Dataset storage
├── models/                # Saved models
├── notebooks/             # Jupyter notebooks for experiments
└── requirements.txt       # Dependencies
```

### Next Steps:
1. Add argparse to train.py for command-line arguments
2. Create config files for hyperparameters
3. Add logging functionality
4. Create visualization utilities
5. Add model evaluation metrics
6. Implement model checkpointing